# 95% Confidence Intervals for SAMVAE Results Tables

This notebook computes and displays **95% confidence intervals** (CI) for all SAMVAE results tables:

1. **Hyperparameter Optimization (HPO)**  
2. **Intermediate Combinations**  
3. **Final Combinations**  

The 95% CI is computed using the t-distribution:

$$\text{CI}_{95\%} = \bar{x} \pm t_{0.975,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

## 1. Configuration & Imports

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import html as _html
from scipy.stats import t as t_dist
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Configuration
Results_dir = os.path.join('..', 'results')
Datasets = ['brca', 'lgg']

## 2. Helper Functions

In [ ]:
def compute_95_ci(values):
    # Compute mean, std, and 95% confidence interval using t-distribution
    n = len(values)
    mean = np.mean(values)
    std = np.std(values, ddof=0)
    if n > 1:
        t_crit = t_dist.ppf(0.975, df=n - 1)
        margin = t_crit * (std / np.sqrt(n))
        ci_low = mean - margin
        ci_up = mean + margin
    else:
        ci_low = ci_up = mean
    return mean, std, ci_low, ci_up, n

def fmt_with_ci(mean, std, ci_low, ci_up):
    # 'mean ± std [low, up]'
    return f"{mean:.3f} ± {std:.3f} [{ci_low:.3f}, {ci_up:.3f}]"

def extract_values_sa(data, key='best_cis'):
    values = []
    for fold in data[key]:
        for seed_list in fold:
            for seed in seed_list:
                values.append(seed[1])
    return values

def extract_values_cr(data, risk_idx, key='best_cis'):
    values = []
    for fold in data[key]:
        for seed_list in fold[risk_idx]:
            for seed in seed_list:
                values.append(seed[1])
    return values

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def style_dataframe(df, title=''):
    if title:
        display(HTML(f'<h4>{_html.escape(str(title))}</h4>'))
    html = df.to_html(index=False, border=1, justify='center')
    display(HTML(html))


## 3. HPO Tables with 95% CI

### 3.1 Survival Analysis — Hyperparameter Optimization

In [ ]:
def recompute_per_param_sa(results, seeds_eval=3):
    all_params = list(results.keys())
    n_seeds = len(results[all_params[0]])
    n_folds = len(results[all_params[0]][0])

    best_results_per_param = []
    for model_params in all_params:
        fold_results = []
        for fold in range(n_folds):
            ci_per_seed = [results[model_params][seed][fold]['ci'][-1] for seed in range(n_seeds)]
            ibs_per_seed = [results[model_params][seed][fold]['ibs'][-1] for seed in range(n_seeds)]

            differences = []
            best_idx = []
            for i in range(len(ci_per_seed)):
                diff = ci_per_seed[i][0][1] - ibs_per_seed[i][0][1]

                if seeds_eval > len(differences):
                    best_idx.append(i)
                    differences.append(diff)
                else:
                    min_dif_idx = np.argmin(differences)
                    if diff > differences[min_dif_idx]:
                        differences[min_dif_idx] = diff
                        best_idx[min_dif_idx] = i

            fold_results.append((
                fold,
                np.mean(np.array([ci[0][1] for ci in ci_per_seed])[best_idx]),
                [ci_per_seed[idx] for idx in best_idx],
                np.mean(np.array([ibs[0][1] for ibs in ibs_per_seed])[best_idx]),
                [ibs_per_seed[idx] for idx in best_idx]
            ))

        avg_ci = sum([x[1] for x in fold_results]) / n_folds
        avg_ibs = sum([x[3] for x in fold_results]) / n_folds

        best_results_per_param.append({
            'param_comb': model_params,
            'avg_ci': avg_ci,
            'avg_ibs': avg_ibs,
            'best_cis': [x[2] for x in fold_results],
            'best_ibs': [x[4] for x in fold_results]
        })

    return best_results_per_param


def get_hpo_table_sa_with_ci(dataset):
    hpo_base = os.path.join(Results_dir, 'Hyperparameter_optimization', 'Survival_Analysis', dataset)
    if not os.path.exists(hpo_base):
        print(f'  No HPO SA results for {dataset}')
        return None

    rows = []

    for modality_dir in sorted(os.listdir(hpo_base)):
        modality_path = os.path.join(hpo_base, modality_dir)
        if not os.path.isdir(modality_path):
            continue
        for config_dir in os.listdir(modality_path):
            config_path = os.path.join(modality_path, config_dir)
            ci_path = os.path.join(config_path, 'best_results.pkl')
            ibs_path = os.path.join(config_path, 'best_results_ibs.pkl')
            if not (os.path.exists(ci_path) and os.path.exists(ibs_path)):
                continue

            ci_data = load_pkl(ci_path)
            ibs_data = load_pkl(ibs_path)

            # Try to load per_param, otherwise recompute from results.pkl
            per_param_path = os.path.join(config_path, 'best_results_per_param.pkl')
            if os.path.exists(per_param_path):
                per_param = load_pkl(per_param_path)
            else:
                results_path = os.path.join(config_path, 'results.pkl')
                if os.path.exists(results_path):
                    results_raw = load_pkl(results_path)
                    per_param = recompute_per_param_sa(results_raw)
                else:
                    per_param = [ci_data]  

            for param_result in (per_param if isinstance(per_param, list) else [per_param]):
                param_comb = param_result.get('param_comb', '-')
                parts = str(param_comb).split('_') if '_' in str(param_comb) else ['-', '-']
                latent = parts[0] if len(parts) >= 1 else '-'
                hidden = parts[1] if len(parts) >= 2 else '-'

                cis_list = extract_values_sa(param_result, 'best_cis')
                ibs_list = extract_values_sa(param_result, 'best_ibs')

                if not cis_list or not ibs_list:
                    continue

                ci_mean, ci_std, ci_lo, ci_hi, n = compute_95_ci(cis_list)
                ibs_mean, ibs_std, ibs_lo, ibs_hi, _ = compute_95_ci(ibs_list)
                diff_list = [c - i for c, i in zip(cis_list, ibs_list)]
                diff_mean, diff_std, diff_lo, diff_hi, _ = compute_95_ci(diff_list)

                rows.append({
                    'Dataset': dataset,
                    'Modes': modality_dir,
                    'Latent': latent,
                    'Hidden': hidden,
                    'C-INDEX (95% CI)': fmt_with_ci(ci_mean, ci_std, ci_lo, ci_hi),
                    'IBS (95% CI)': fmt_with_ci(ibs_mean, ibs_std, ibs_lo, ibs_hi),
                    'CI-IBS (95% CI)': fmt_with_ci(diff_mean, diff_std, diff_lo, diff_hi),
                })

    if rows:
        return pd.DataFrame(rows)
    return None

for ds in Datasets:
    df = get_hpo_table_sa_with_ci(ds)
    if df is not None:
        style_dataframe(df, f'HPO Survival Analysis — {ds.upper()}')
    else:
        print(f'No HPO SA results for {ds}')

Dataset,Modes,Latent,Hidden,C-INDEX (95% CI),IBS (95% CI),CI-IBS (95% CI)
brca,clinical,5,5,"0.640 ± 0.061 [0.606, 0.674]","0.195 ± 0.027 [0.180, 0.209]","0.445 ± 0.072 [0.405, 0.485]"
brca,clinical,5,10,"0.698 ± 0.047 [0.672, 0.724]","0.188 ± 0.037 [0.167, 0.209]","0.510 ± 0.070 [0.471, 0.549]"
brca,clinical,5,25,"0.678 ± 0.066 [0.641, 0.715]","0.187 ± 0.026 [0.173, 0.202]","0.491 ± 0.073 [0.450, 0.531]"
brca,clinical,5,50,"0.711 ± 0.061 [0.677, 0.745]","0.182 ± 0.032 [0.165, 0.200]","0.529 ± 0.080 [0.484, 0.573]"
brca,clinical,5,75,"0.707 ± 0.056 [0.676, 0.738]","0.191 ± 0.034 [0.172, 0.210]","0.516 ± 0.068 [0.478, 0.554]"
brca,clinical,5,100,"0.699 ± 0.062 [0.664, 0.733]","0.185 ± 0.033 [0.166, 0.203]","0.514 ± 0.087 [0.466, 0.562]"
brca,clinical,5,500,"0.720 ± 0.058 [0.687, 0.752]","0.182 ± 0.022 [0.169, 0.194]","0.538 ± 0.074 [0.497, 0.579]"
brca,clinical,10,5,"0.654 ± 0.066 [0.617, 0.690]","0.192 ± 0.033 [0.174, 0.211]","0.462 ± 0.088 [0.413, 0.511]"
brca,clinical,10,10,"0.692 ± 0.055 [0.661, 0.722]","0.185 ± 0.027 [0.170, 0.200]","0.506 ± 0.073 [0.466, 0.547]"
brca,clinical,10,25,"0.718 ± 0.054 [0.688, 0.748]","0.181 ± 0.031 [0.164, 0.199]","0.537 ± 0.070 [0.498, 0.575]"


Dataset,Modes,Latent,Hidden,C-INDEX (95% CI),IBS (95% CI),CI-IBS (95% CI)
lgg,clinical,5,5,"0.709 ± 0.072 [0.670, 0.749]","0.195 ± 0.034 [0.176, 0.213]","0.515 ± 0.096 [0.462, 0.568]"
lgg,clinical,5,10,"0.754 ± 0.064 [0.719, 0.790]","0.186 ± 0.023 [0.173, 0.199]","0.568 ± 0.076 [0.526, 0.610]"
lgg,clinical,5,25,"0.764 ± 0.069 [0.726, 0.802]","0.180 ± 0.018 [0.170, 0.191]","0.584 ± 0.070 [0.545, 0.622]"
lgg,clinical,5,50,"0.766 ± 0.061 [0.732, 0.800]","0.183 ± 0.019 [0.173, 0.194]","0.583 ± 0.065 [0.547, 0.619]"
lgg,clinical,5,75,"0.772 ± 0.062 [0.737, 0.806]","0.179 ± 0.022 [0.167, 0.191]","0.593 ± 0.069 [0.555, 0.631]"
lgg,clinical,5,100,"0.767 ± 0.052 [0.738, 0.796]","0.180 ± 0.022 [0.168, 0.192]","0.587 ± 0.060 [0.554, 0.620]"
lgg,clinical,5,500,"0.763 ± 0.055 [0.732, 0.793]","0.178 ± 0.021 [0.166, 0.189]","0.585 ± 0.063 [0.550, 0.620]"
lgg,clinical,10,5,"0.739 ± 0.082 [0.694, 0.785]","0.193 ± 0.026 [0.178, 0.208]","0.546 ± 0.096 [0.493, 0.599]"
lgg,clinical,10,10,"0.769 ± 0.062 [0.734, 0.803]","0.174 ± 0.017 [0.165, 0.184]","0.595 ± 0.069 [0.557, 0.633]"
lgg,clinical,10,25,"0.771 ± 0.057 [0.739, 0.803]","0.176 ± 0.016 [0.167, 0.185]","0.594 ± 0.059 [0.562, 0.627]"


### 3.2 Competing Risks — Hyperparameter Optimization

In [ ]:
def recompute_per_param_cr(results_cr, seeds_eval=3):
    all_params = list(results_cr.keys())
    n_seeds = len(results_cr[all_params[0]])
    n_folds = len(results_cr[all_params[0]][0])

    best_results_per_param = []
    for model_params in all_params:
        fold_results = []
        for fold in range(n_folds):
            ci_per_seed = [results_cr[model_params][seed][fold]['ci'][-1] for seed in range(n_seeds)]
            ibs_per_seed = [results_cr[model_params][seed][fold]['ibs'][-1] for seed in range(n_seeds)]

            num_tensors = len(ci_per_seed[0])  # number of risks
            best_cis_per_tensor = []
            best_ibs_per_tensor = []

            for tensor_idx in range(num_tensors):
                cis_tensor = [np.mean(ci_per_seed[seed][tensor_idx]) for seed in range(n_seeds)]
                ibs_tensor = [np.mean(ibs_per_seed[seed][tensor_idx]) for seed in range(n_seeds)]

                best_cis_idx = np.argsort(cis_tensor)[-seeds_eval:][::-1]
                best_ibs_idx = np.argsort(ibs_tensor)[:seeds_eval]

                best_cis_per_tensor.append([ci_per_seed[idx][tensor_idx] for idx in best_cis_idx])
                best_ibs_per_tensor.append([ibs_per_seed[idx][tensor_idx] for idx in best_ibs_idx])

            fold_results.append((fold,
                [[t[1] for t in ci] for ci in ci_per_seed[0]],
                best_cis_per_tensor,
                [[t[1] for t in ibs] for ibs in ibs_per_seed[0]],
                best_ibs_per_tensor
            ))

        avg_ci = [sum(ci) / n_folds for ci in zip(*[sum(x[1], []) for x in fold_results])]
        avg_ibs = [sum(ibs) / n_folds for ibs in zip(*[sum(x[3], []) for x in fold_results])]

        best_results_per_param.append({
            'param_comb': model_params,
            'avg_ci': avg_ci,
            'avg_ibs': avg_ibs,
            'best_cis': [x[2] for x in fold_results],
            'best_ibs': [x[4] for x in fold_results]
        })
    return best_results_per_param


def get_hpo_table_cr_with_ci(dataset):
    hpo_base = os.path.join(Results_dir, 'Hyperparameter_optimization', 'Competing_Risks', dataset)
    if not os.path.exists(hpo_base):
        print(f'  No HPO CR results for {dataset}')
        return None

    # Load labels
    labels_path = os.path.join(hpo_base, 'labels.pkl')
    if os.path.exists(labels_path):
        labels = load_pkl(labels_path)
    else:
        labels = None

    rows = []

    for modality_dir in sorted(os.listdir(hpo_base)):
        modality_path = os.path.join(hpo_base, modality_dir)
        if not os.path.isdir(modality_path):
            continue
        for config_dir in os.listdir(modality_path):
            config_path = os.path.join(modality_path, config_dir)

            # Determine number of risks from best_results_cr
            cr_path = os.path.join(config_path, 'best_results_cr.pkl')
            ibs_cr_path = os.path.join(config_path, 'best_results_ibs_cr.pkl')
            if not (os.path.exists(cr_path) and os.path.exists(ibs_cr_path)):
                continue

            ci_data = load_pkl(cr_path)
            num_risks = len(ci_data['best_cis'][0]) if ci_data['best_cis'] else 0
            risk_labels = labels if labels else [str(i) for i in range(num_risks)]

            # Try to load per_param, otherwise recompute from results_cr.pkl
            per_param_path = os.path.join(config_path, 'best_results_per_param.pkl')
            if os.path.exists(per_param_path):
                per_param = load_pkl(per_param_path)
            else:
                results_cr_path = os.path.join(config_path, 'results_cr.pkl')
                if not os.path.exists(results_cr_path):
                    results_cr_path = os.path.join(config_path, 'results.pkl')
                if os.path.exists(results_cr_path):
                    results_cr = load_pkl(results_cr_path)
                    per_param = recompute_per_param_cr(results_cr)
                else:
                    per_param = [ci_data] 

            for param_result in (per_param if isinstance(per_param, list) else [per_param]):
                param_comb = param_result.get('param_comb', '-')
                parts = str(param_comb).split('_') if '_' in str(param_comb) else ['-', '-']
                latent = parts[0] if len(parts) >= 1 else '-'
                hidden = parts[1] if len(parts) >= 2 else '-'

                avg_ci_all_risks = []
                avg_ibs_all_risks = []
                temp_risk_data = []

                for risk_idx in range(num_risks):
                    cis_list = extract_values_cr(param_result, risk_idx, 'best_cis')
                    ibs_list = extract_values_cr(param_result, risk_idx, 'best_ibs')

                    if not cis_list or not ibs_list:
                        continue

                    ci_mean, ci_std, ci_lo, ci_hi, n = compute_95_ci(cis_list)
                    ibs_mean, ibs_std, ibs_lo, ibs_hi, _ = compute_95_ci(ibs_list)
                    avg_ci_all_risks.append(ci_mean)
                    avg_ibs_all_risks.append(ibs_mean)

                    temp_risk_data.append({
                        'row_data': {
                            'Dataset': dataset,
                            'Modes': modality_dir,
                            'Latent': latent,
                            'Hidden': hidden,
                            'Risk': risk_labels[risk_idx] if risk_idx < len(risk_labels) else str(risk_idx),
                            'C-INDEX (95% CI)': fmt_with_ci(ci_mean, ci_std, ci_lo, ci_hi),
                            'IBS (95% CI)': fmt_with_ci(ibs_mean, ibs_std, ibs_lo, ibs_hi),
                            'AVG CI': None,
                            'AVG IBS': None,
                            'CI-IBS': None,
                        }
                    })

                # Compute AVG CI, AVG IBS, CI-IBS across risks
                if avg_ci_all_risks and avg_ibs_all_risks:
                    avg_ci_global = np.mean(avg_ci_all_risks)
                    avg_ibs_global = np.mean(avg_ibs_all_risks)
                    diff_global = avg_ci_global - avg_ibs_global
                    for tr in temp_risk_data:
                        tr['row_data']['AVG CI'] = f"{avg_ci_global:.3f}"
                        tr['row_data']['AVG IBS'] = f"{avg_ibs_global:.3f}"
                        tr['row_data']['CI-IBS'] = f"{diff_global:.3f}"

                for tr in temp_risk_data:
                    rows.append(tr['row_data'])

    if rows:
        return pd.DataFrame(rows)
    return None


for ds in Datasets:
    df = get_hpo_table_cr_with_ci(ds)
    if df is not None:
        style_dataframe(df, f'HPO Competing Risks — {ds.upper()}')
    else:
        print(f'No HPO CR results for {ds}')

Dataset,Modes,Latent,Hidden,Risk,C-INDEX (95% CI),IBS (95% CI),AVG CI,AVG IBS,CI-IBS
brca,clinical,5,5,0,"0.636 ± 0.072 [0.596, 0.676]","0.210 ± 0.045 [0.186, 0.235]",0.640,0.232,0.408
brca,clinical,5,5,1,"0.644 ± 0.079 [0.600, 0.688]","0.254 ± 0.065 [0.218, 0.290]",0.640,0.232,0.408
brca,clinical,5,10,0,"0.597 ± 0.052 [0.568, 0.626]","0.259 ± 0.051 [0.230, 0.287]",0.622,0.224,0.398
brca,clinical,5,10,1,"0.647 ± 0.052 [0.618, 0.676]","0.190 ± 0.044 [0.166, 0.215]",0.622,0.224,0.398
brca,clinical,5,25,0,"0.640 ± 0.057 [0.609, 0.672]","0.249 ± 0.058 [0.217, 0.281]",0.637,0.229,0.408
brca,clinical,5,25,1,"0.633 ± 0.049 [0.606, 0.660]","0.208 ± 0.040 [0.186, 0.230]",0.637,0.229,0.408
brca,clinical,5,50,0,"0.625 ± 0.041 [0.602, 0.647]","0.238 ± 0.043 [0.214, 0.262]",0.611,0.225,0.386
brca,clinical,5,50,1,"0.597 ± 0.044 [0.573, 0.622]","0.212 ± 0.056 [0.180, 0.243]",0.611,0.225,0.386
brca,clinical,5,75,0,"0.600 ± 0.045 [0.575, 0.625]","0.221 ± 0.053 [0.192, 0.251]",0.616,0.225,0.391
brca,clinical,5,75,1,"0.632 ± 0.058 [0.600, 0.664]","0.229 ± 0.036 [0.209, 0.249]",0.616,0.225,0.391


Dataset,Modes,Latent,Hidden,Risk,C-INDEX (95% CI),IBS (95% CI),AVG CI,AVG IBS,CI-IBS
lgg,clinical,5,5,0,"0.588 ± 0.032 [0.571, 0.606]","0.268 ± 0.081 [0.223, 0.313]",0.575,0.295,0.279
lgg,clinical,5,5,1,"0.561 ± 0.026 [0.547, 0.576]","0.323 ± 0.091 [0.272, 0.373]",0.575,0.295,0.279
lgg,clinical,5,10,0,"0.592 ± 0.045 [0.567, 0.616]","0.290 ± 0.078 [0.247, 0.334]",0.578,0.297,0.281
lgg,clinical,5,10,1,"0.564 ± 0.033 [0.546, 0.582]","0.304 ± 0.081 [0.259, 0.349]",0.578,0.297,0.281
lgg,clinical,5,25,0,"0.569 ± 0.032 [0.551, 0.586]","0.281 ± 0.064 [0.245, 0.316]",0.571,0.287,0.284
lgg,clinical,5,25,1,"0.573 ± 0.029 [0.557, 0.589]","0.292 ± 0.061 [0.258, 0.326]",0.571,0.287,0.284
lgg,clinical,5,50,0,"0.611 ± 0.052 [0.582, 0.639]","0.264 ± 0.068 [0.226, 0.302]",0.598,0.295,0.303
lgg,clinical,5,50,1,"0.585 ± 0.044 [0.561, 0.609]","0.326 ± 0.082 [0.281, 0.372]",0.598,0.295,0.303
lgg,clinical,5,75,0,"0.606 ± 0.033 [0.588, 0.625]","0.266 ± 0.067 [0.228, 0.303]",0.597,0.280,0.317
lgg,clinical,5,75,1,"0.588 ± 0.046 [0.562, 0.614]","0.295 ± 0.080 [0.251, 0.340]",0.597,0.280,0.317


## 4. Intermediate Combinations with 95% CI

### 4.1 Survival Analysis — Intermediate Combinations

In [ ]:
def get_combinations_table_sa_with_ci(category, dataset):
    base = os.path.join(Results_dir, category, 'Survival_Analysis', dataset)
    if not os.path.exists(base):
        print(f'  No {category} SA results for {dataset}')
        return None

    rows = []

    for root, dirs, files in os.walk(base):
        if 'best_results.pkl' in files and 'best_results_ibs.pkl' in files:
            ci_data = load_pkl(os.path.join(root, 'best_results.pkl'))
            ibs_data = load_pkl(os.path.join(root, 'best_results_ibs.pkl'))

            cis_list = extract_values_sa(ci_data, 'best_cis')
            ibs_list = extract_values_sa(ibs_data, 'best_ibs')

            if not cis_list or not ibs_list:
                continue

            ci_mean, ci_std, ci_lo, ci_hi, n = compute_95_ci(cis_list)
            ibs_mean, ibs_std, ibs_lo, ibs_hi, _ = compute_95_ci(ibs_list)
            diff_list = [c - i for c, i in zip(cis_list, ibs_list)]
            diff_mean, diff_std, diff_lo, diff_hi, _ = compute_95_ci(diff_list)

            relative_path = os.path.relpath(root, base)
            mode_name = os.path.dirname(relative_path)

            rows.append({
                'Dataset': dataset,
                'Modes': mode_name,
                'C-INDEX (95% CI)': fmt_with_ci(ci_mean, ci_std, ci_lo, ci_hi),
                'IBS (95% CI)': fmt_with_ci(ibs_mean, ibs_std, ibs_lo, ibs_hi),
                'CI-IBS (95% CI)': fmt_with_ci(diff_mean, diff_std, diff_lo, diff_hi),
            })

    if rows:
        df = pd.DataFrame(rows)
        df['_sort'] = df['CI-IBS (95% CI)'].apply(lambda x: float(x.split(' ±')[0]))
        df = df.sort_values('_sort', ascending=False).drop(columns='_sort').reset_index(drop=True)
        return df
    return None


for ds in Datasets:
    df = get_combinations_table_sa_with_ci('Intermediate_Combinations', ds)
    if df is not None:
        style_dataframe(df, f'Intermediate Combinations — Survival Analysis — {ds.upper()}')
    else:
        print(f'No Intermediate Combinations SA results for {ds}')

Dataset,Modes,C-INDEX (95% CI),IBS (95% CI),CI-IBS (95% CI)
brca,clinical_omic_cnv,"0.684 ± 0.053 [0.654, 0.713]","0.175 ± 0.034 [0.156, 0.194]","0.509 ± 0.069 [0.471, 0.547]"
brca,clinical_omic_miRNA,"0.687 ± 0.072 [0.647, 0.726]","0.207 ± 0.039 [0.186, 0.229]","0.480 ± 0.084 [0.433, 0.526]"
brca,clinical_wsi_patches_15_patch,"0.663 ± 0.049 [0.636, 0.690]","0.194 ± 0.025 [0.180, 0.208]","0.469 ± 0.048 [0.443, 0.495]"
brca,clinical_omic_RNAseq,"0.654 ± 0.051 [0.626, 0.682]","0.186 ± 0.020 [0.175, 0.197]","0.468 ± 0.043 [0.444, 0.493]"
brca,clinical_omic_adn,"0.661 ± 0.071 [0.621, 0.700]","0.200 ± 0.042 [0.177, 0.223]","0.461 ± 0.086 [0.413, 0.508]"
brca,clinical_omic_adn_omic_RNAseq,"0.628 ± 0.039 [0.606, 0.650]","0.186 ± 0.019 [0.175, 0.196]","0.442 ± 0.046 [0.417, 0.468]"
brca,clinical_wsi_patches_5_patch,"0.637 ± 0.065 [0.601, 0.673]","0.198 ± 0.033 [0.180, 0.217]","0.439 ± 0.085 [0.392, 0.486]"
brca,clinical_omic_miRNA_omic_RNAseq,"0.629 ± 0.073 [0.589, 0.669]","0.193 ± 0.028 [0.177, 0.208]","0.436 ± 0.086 [0.388, 0.484]"
brca,clinical_wsi_patches_1_patch,"0.638 ± 0.063 [0.603, 0.673]","0.203 ± 0.029 [0.186, 0.219]","0.435 ± 0.076 [0.393, 0.477]"
brca,clinical_wsi_patches_10_patch,"0.633 ± 0.054 [0.603, 0.662]","0.198 ± 0.026 [0.183, 0.212]","0.435 ± 0.053 [0.406, 0.464]"


Dataset,Modes,C-INDEX (95% CI),IBS (95% CI),CI-IBS (95% CI)
lgg,clinical_wsi_patches_15_patch,"0.761 ± 0.062 [0.727, 0.796]","0.175 ± 0.025 [0.161, 0.189]","0.586 ± 0.081 [0.541, 0.631]"
lgg,clinical_wsi_patches_10_patch,"0.773 ± 0.043 [0.749, 0.796]","0.187 ± 0.020 [0.175, 0.198]","0.586 ± 0.050 [0.558, 0.614]"
lgg,clinical_wsi_patches_5_patch,"0.768 ± 0.048 [0.741, 0.794]","0.187 ± 0.023 [0.174, 0.200]","0.581 ± 0.061 [0.547, 0.614]"
lgg,clinical_omic_miRNA,"0.763 ± 0.041 [0.740, 0.785]","0.187 ± 0.017 [0.178, 0.196]","0.576 ± 0.043 [0.552, 0.600]"
lgg,clinical_omic_adn_omic_RNAseq,"0.751 ± 0.059 [0.719, 0.784]","0.185 ± 0.021 [0.174, 0.197]","0.566 ± 0.066 [0.529, 0.603]"
lgg,clinical_omic_adn,"0.754 ± 0.050 [0.727, 0.782]","0.190 ± 0.027 [0.175, 0.204]","0.565 ± 0.060 [0.531, 0.598]"
lgg,clinical_omic_RNAseq,"0.750 ± 0.051 [0.721, 0.778]","0.187 ± 0.016 [0.178, 0.196]","0.563 ± 0.054 [0.533, 0.592]"
lgg,clinical_omic_cnv_omic_RNAseq,"0.751 ± 0.056 [0.720, 0.781]","0.188 ± 0.024 [0.175, 0.201]","0.563 ± 0.061 [0.529, 0.597]"
lgg,clinical_omic_cnv,"0.736 ± 0.052 [0.707, 0.765]","0.186 ± 0.017 [0.177, 0.196]","0.549 ± 0.053 [0.520, 0.579]"
lgg,clinical_omic_miRNA_omic_RNAseq,"0.733 ± 0.055 [0.703, 0.763]","0.185 ± 0.011 [0.179, 0.192]","0.547 ± 0.055 [0.517, 0.578]"


### 4.2 Competing Risks — Intermediate Combinations

In [ ]:
def get_combinations_table_cr_with_ci(category, dataset):
    base = os.path.join(Results_dir, category, 'Competing_Risks', dataset)
    if not os.path.exists(base):
        print(f'  No {category} CR results for {dataset}')
        return None

    # Load labels
    labels_path = os.path.join(base, 'labels.pkl')
    labels = load_pkl(labels_path) if os.path.exists(labels_path) else None

    rows = []
    ci_file = 'best_results_cr.pkl'
    ibs_file = 'best_results_ibs_cr.pkl'

    for root, dirs, files in os.walk(base):
        if ci_file in files and ibs_file in files:
            ci_data = load_pkl(os.path.join(root, ci_file))
            ibs_data = load_pkl(os.path.join(root, ibs_file))

            num_risks = len(ci_data['best_cis'][0]) if ci_data['best_cis'] else 0
            risk_labels = labels if labels else [str(i) for i in range(num_risks)]

            relative_path = os.path.relpath(root, base)
            mode_name = os.path.dirname(relative_path)

            avg_ci_all, avg_ibs_all = [], []
            risk_rows = []

            for risk_idx in range(num_risks):
                cis_list = extract_values_cr(ci_data, risk_idx, 'best_cis')
                ibs_list = extract_values_cr(ibs_data, risk_idx, 'best_ibs')

                if not cis_list or not ibs_list:
                    continue

                ci_mean, ci_std, ci_lo, ci_hi, n = compute_95_ci(cis_list)
                ibs_mean, ibs_std, ibs_lo, ibs_hi, _ = compute_95_ci(ibs_list)
                avg_ci_all.append(ci_mean)
                avg_ibs_all.append(ibs_mean)

                risk_rows.append({
                    'Dataset': dataset,
                    'Modes': mode_name,
                    'Risk': risk_labels[risk_idx] if risk_idx < len(risk_labels) else str(risk_idx),
                    'C-INDEX (95% CI)': fmt_with_ci(ci_mean, ci_std, ci_lo, ci_hi),
                    'IBS (95% CI)': fmt_with_ci(ibs_mean, ibs_std, ibs_lo, ibs_hi),
                    'AVG CI': None,
                    'AVG IBS': None,
                    'CI-IBS': None,
                })

            if avg_ci_all and avg_ibs_all:
                mean_ci_global = np.mean(avg_ci_all)
                mean_ibs_global = np.mean(avg_ibs_all)
                diff_global = mean_ci_global - mean_ibs_global
                for r in risk_rows:
                    r['AVG CI'] = f"{mean_ci_global:.3f}"
                    r['AVG IBS'] = f"{mean_ibs_global:.3f}"
                    r['CI-IBS'] = f"{diff_global:.3f}"

            rows.extend(risk_rows)

    if rows:
        return pd.DataFrame(rows)
    return None


for ds in Datasets:
    df = get_combinations_table_cr_with_ci('Intermediate_Combinations', ds)
    if df is not None:
        style_dataframe(df, f'Intermediate Combinations — Competing Risks — {ds.upper()}')
    else:
        print(f'No Intermediate Combinations CR results for {ds}')

Dataset,Modes,Risk,C-INDEX (95% CI),IBS (95% CI),AVG CI,AVG IBS,CI-IBS
brca,clinical_omic_adn_omic_cnv_omic_RNAseq,0,"0.596 ± 0.055 [0.566, 0.627]","0.211 ± 0.044 [0.186, 0.235]",0.588,0.222,0.366
brca,clinical_omic_adn_omic_cnv_omic_RNAseq,1,"0.580 ± 0.031 [0.563, 0.597]","0.234 ± 0.065 [0.198, 0.269]",0.588,0.222,0.366
brca,clinical_omic_adn_omic_miRNA,0,"0.610 ± 0.061 [0.576, 0.643]","0.230 ± 0.040 [0.208, 0.253]",0.609,0.239,0.370
brca,clinical_omic_adn_omic_miRNA,1,"0.607 ± 0.055 [0.577, 0.638]","0.247 ± 0.052 [0.218, 0.275]",0.609,0.239,0.370
brca,clinical_omic_adn_omic_cnv_omic_miRNA_omic_RNAseq,0,"0.624 ± 0.071 [0.585, 0.663]","0.214 ± 0.069 [0.176, 0.253]",0.609,0.227,0.382
brca,clinical_omic_adn_omic_cnv_omic_miRNA_omic_RNAseq,1,"0.594 ± 0.053 [0.565, 0.623]","0.241 ± 0.040 [0.218, 0.263]",0.609,0.227,0.382
brca,clinical_omic_adn_omic_miRNA_omic_RNAseq,0,"0.620 ± 0.040 [0.598, 0.642]","0.223 ± 0.056 [0.192, 0.254]",0.624,0.229,0.395
brca,clinical_omic_adn_omic_miRNA_omic_RNAseq,1,"0.628 ± 0.055 [0.598, 0.659]","0.234 ± 0.052 [0.206, 0.263]",0.624,0.229,0.395
brca,clinical_omic_adn,0,"0.632 ± 0.083 [0.586, 0.678]","0.221 ± 0.064 [0.186, 0.257]",0.625,0.229,0.396
brca,clinical_omic_adn,1,"0.617 ± 0.055 [0.587, 0.648]","0.237 ± 0.047 [0.211, 0.263]",0.625,0.229,0.396


Dataset,Modes,Risk,C-INDEX (95% CI),IBS (95% CI),AVG CI,AVG IBS,CI-IBS
lgg,clinical_omic_adn_omic_cnv_omic_RNAseq,0,"0.546 ± 0.024 [0.533, 0.560]","0.283 ± 0.053 [0.253, 0.312]",0.557,0.283,0.274
lgg,clinical_omic_adn_omic_cnv_omic_RNAseq,1,"0.568 ± 0.030 [0.552, 0.585]","0.283 ± 0.073 [0.243, 0.323]",0.557,0.283,0.274
lgg,clinical_omic_adn_omic_miRNA,0,"0.559 ± 0.027 [0.544, 0.574]","0.277 ± 0.050 [0.249, 0.305]",0.562,0.302,0.260
lgg,clinical_omic_adn_omic_miRNA,1,"0.564 ± 0.020 [0.553, 0.575]","0.326 ± 0.066 [0.290, 0.363]",0.562,0.302,0.260
lgg,clinical_omic_adn_omic_cnv_omic_miRNA_omic_RNAseq,0,"0.546 ± 0.024 [0.533, 0.559]","0.282 ± 0.065 [0.246, 0.318]",0.548,0.301,0.247
lgg,clinical_omic_adn_omic_cnv_omic_miRNA_omic_RNAseq,1,"0.550 ± 0.027 [0.535, 0.565]","0.320 ± 0.064 [0.284, 0.356]",0.548,0.301,0.247
lgg,clinical_omic_adn_omic_miRNA_omic_RNAseq,0,"0.574 ± 0.029 [0.558, 0.590]","0.292 ± 0.087 [0.244, 0.340]",0.570,0.293,0.278
lgg,clinical_omic_adn_omic_miRNA_omic_RNAseq,1,"0.567 ± 0.045 [0.542, 0.592]","0.294 ± 0.092 [0.243, 0.345]",0.570,0.293,0.278
lgg,clinical_omic_adn,0,"0.566 ± 0.028 [0.550, 0.582]","0.288 ± 0.061 [0.255, 0.322]",0.564,0.291,0.273
lgg,clinical_omic_adn,1,"0.562 ± 0.034 [0.543, 0.581]","0.293 ± 0.064 [0.258, 0.329]",0.564,0.291,0.273


## 5. Final Combinations with 95% CI

### 5.1 Survival Analysis — Final Combinations

In [ ]:
for ds in Datasets:
    df = get_combinations_table_sa_with_ci('Final_Combinations', ds)
    if df is not None:
        style_dataframe(df, f'Final Combinations — Survival Analysis — {ds.upper()}')
    else:
        print(f'No Final Combinations SA results for {ds}')

Dataset,Modes,C-INDEX (95% CI),IBS (95% CI),CI-IBS (95% CI)
brca,clinical,"0.722 ± 0.057 [0.690, 0.754]","0.175 ± 0.025 [0.162, 0.189]","0.546 ± 0.066 [0.510, 0.583]"
brca,clinical_omic_cnv,"0.705 ± 0.061 [0.672, 0.739]","0.191 ± 0.035 [0.172, 0.211]","0.514 ± 0.077 [0.471, 0.557]"
brca,clinical_wsi_patches_15_patch,"0.699 ± 0.046 [0.674, 0.725]","0.192 ± 0.019 [0.181, 0.203]","0.507 ± 0.054 [0.477, 0.537]"
brca,clinical_omic_cnv_wsi_patches_15_patch,"0.688 ± 0.060 [0.655, 0.722]","0.192 ± 0.019 [0.182, 0.203]","0.496 ± 0.057 [0.465, 0.528]"
brca,omic_cnv_wsi_patches_15_patch,"0.640 ± 0.060 [0.607, 0.673]","0.196 ± 0.021 [0.184, 0.208]","0.444 ± 0.059 [0.411, 0.477]"
brca,omic_cnv,"0.636 ± 0.034 [0.617, 0.655]","0.195 ± 0.020 [0.184, 0.206]","0.441 ± 0.031 [0.423, 0.458]"
brca,wsi_patches_15_patch,"0.615 ± 0.074 [0.574, 0.656]","0.203 ± 0.029 [0.188, 0.219]","0.411 ± 0.085 [0.365, 0.458]"


Dataset,Modes,C-INDEX (95% CI),IBS (95% CI),CI-IBS (95% CI)
lgg,clinical_wsi_patches_10_patch,"0.773 ± 0.043 [0.749, 0.796]","0.187 ± 0.020 [0.175, 0.198]","0.586 ± 0.050 [0.558, 0.614]"
lgg,clinical_omic_adn_wsi_patches_10_patch,"0.761 ± 0.047 [0.736, 0.787]","0.186 ± 0.023 [0.173, 0.198]","0.576 ± 0.052 [0.547, 0.604]"
lgg,clinical,"0.773 ± 0.089 [0.724, 0.823]","0.201 ± 0.035 [0.182, 0.221]","0.572 ± 0.084 [0.526, 0.618]"
lgg,clinical_omic_adn,"0.754 ± 0.050 [0.727, 0.782]","0.190 ± 0.027 [0.175, 0.204]","0.565 ± 0.060 [0.531, 0.598]"
lgg,omic_adn_wsi_patches_10_patch,"0.620 ± 0.059 [0.587, 0.652]","0.198 ± 0.019 [0.188, 0.209]","0.421 ± 0.071 [0.382, 0.461]"
lgg,omic_adn,"0.615 ± 0.042 [0.592, 0.638]","0.196 ± 0.014 [0.188, 0.204]","0.420 ± 0.049 [0.392, 0.447]"
lgg,wsi_patches_10_patch,"0.592 ± 0.029 [0.576, 0.607]","0.195 ± 0.015 [0.187, 0.203]","0.397 ± 0.037 [0.376, 0.417]"


### 5.2 Competing Risks — Final Combinations

In [ ]:
for ds in Datasets:
    df = get_combinations_table_cr_with_ci('Final_Combinations', ds)
    if df is not None:
        style_dataframe(df, f'Final Combinations — Competing Risks — {ds.upper()}')
    else:
        print(f'No Final Combinations CR results for {ds}')

Dataset,Modes,Risk,C-INDEX (95% CI),IBS (95% CI),AVG CI,AVG IBS,CI-IBS
brca,clinical_omic_cnv_omic_RNAseq_wsi_patches_10_patch,0,"0.666 ± 0.066 [0.630, 0.703]","0.183 ± 0.037 [0.163, 0.204]",0.652,0.194,0.458
brca,clinical_omic_cnv_omic_RNAseq_wsi_patches_10_patch,1,"0.637 ± 0.031 [0.620, 0.654]","0.205 ± 0.032 [0.187, 0.222]",0.652,0.194,0.458
brca,omic_cnv_omic_RNAseq_wsi_patches_10_patch,0,"0.641 ± 0.038 [0.620, 0.662]","0.204 ± 0.033 [0.185, 0.222]",0.640,0.207,0.433
brca,omic_cnv_omic_RNAseq_wsi_patches_10_patch,1,"0.639 ± 0.044 [0.615, 0.663]","0.210 ± 0.042 [0.187, 0.233]",0.640,0.207,0.433
brca,clinical,0,"0.664 ± 0.040 [0.642, 0.686]","0.199 ± 0.035 [0.179, 0.218]",0.673,0.202,0.471
brca,clinical,1,"0.682 ± 0.042 [0.659, 0.706]","0.205 ± 0.033 [0.186, 0.223]",0.673,0.202,0.471
brca,wsi_patches_10_patch,0,"0.638 ± 0.037 [0.617, 0.658]","0.206 ± 0.030 [0.189, 0.223]",0.637,0.203,0.433
brca,wsi_patches_10_patch,1,"0.635 ± 0.039 [0.614, 0.657]","0.200 ± 0.022 [0.188, 0.213]",0.637,0.203,0.433
brca,clinical_wsi_patches_10_patch,0,"0.657 ± 0.060 [0.624, 0.690]","0.196 ± 0.023 [0.184, 0.209]",0.648,0.202,0.446
brca,clinical_wsi_patches_10_patch,1,"0.639 ± 0.026 [0.624, 0.653]","0.207 ± 0.021 [0.196, 0.219]",0.648,0.202,0.446


Dataset,Modes,Risk,C-INDEX (95% CI),IBS (95% CI),AVG CI,AVG IBS,CI-IBS
lgg,omic_adn_omic_cnv_omic_miRNA,0,"0.585 ± 0.028 [0.569, 0.600]","0.230 ± 0.047 [0.204, 0.256]",0.587,0.235,0.352
lgg,omic_adn_omic_cnv_omic_miRNA,1,"0.590 ± 0.032 [0.572, 0.608]","0.240 ± 0.039 [0.219, 0.262]",0.587,0.235,0.352
lgg,clinical_omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_patch,0,"0.577 ± 0.022 [0.565, 0.589]","0.233 ± 0.040 [0.210, 0.255]",0.573,0.250,0.323
lgg,clinical_omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_patch,1,"0.570 ± 0.027 [0.555, 0.585]","0.268 ± 0.047 [0.242, 0.294]",0.573,0.250,0.323
lgg,omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_patch,0,"0.579 ± 0.024 [0.565, 0.592]","0.232 ± 0.057 [0.201, 0.264]",0.585,0.243,0.341
lgg,omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_patch,1,"0.591 ± 0.035 [0.571, 0.610]","0.254 ± 0.022 [0.242, 0.266]",0.585,0.243,0.341
lgg,wsi_patches_5_patch,0,"0.585 ± 0.037 [0.565, 0.605]","0.227 ± 0.042 [0.204, 0.250]",0.590,0.233,0.357
lgg,wsi_patches_5_patch,1,"0.595 ± 0.064 [0.560, 0.631]","0.239 ± 0.042 [0.215, 0.262]",0.590,0.233,0.357
lgg,clinical,0,"0.610 ± 0.044 [0.585, 0.634]","0.240 ± 0.046 [0.215, 0.266]",0.613,0.229,0.384
lgg,clinical,1,"0.616 ± 0.045 [0.591, 0.641]","0.217 ± 0.018 [0.207, 0.227]",0.613,0.229,0.384


## 6. Export Tables to CSV

Save all tables with 95% CI to CSV files for easy inclusion in papers or supplementary materials.

In [ ]:
output_dir = os.path.join('figs', 'tables_CI')
os.makedirs(output_dir, exist_ok=True)

for ds in Datasets:
    # HPO SA
    df = get_hpo_table_sa_with_ci(ds)
    if df is not None:
        path = os.path.join(output_dir, f'hpo_sa_{ds}_95ci.csv')
        df.to_csv(path, index=False)
        print(f'Saved: {path}')

    # HPO CR
    df = get_hpo_table_cr_with_ci(ds)
    if df is not None:
        path = os.path.join(output_dir, f'hpo_cr_{ds}_95ci.csv')
        df.to_csv(path, index=False)
        print(f'Saved: {path}')

    # Intermediate SA
    df = get_combinations_table_sa_with_ci('Intermediate_Combinations', ds)
    if df is not None:
        path = os.path.join(output_dir, f'intermediate_sa_{ds}_95ci.csv')
        df.to_csv(path, index=False)
        print(f'Saved: {path}')

    # Intermediate CR
    df = get_combinations_table_cr_with_ci('Intermediate_Combinations', ds)
    if df is not None:
        path = os.path.join(output_dir, f'intermediate_cr_{ds}_95ci.csv')
        df.to_csv(path, index=False)
        print(f'Saved: {path}')

    # Final SA
    df = get_combinations_table_sa_with_ci('Final_Combinations', ds)
    if df is not None:
        path = os.path.join(output_dir, f'final_sa_{ds}_95ci.csv')
        df.to_csv(path, index=False)
        print(f'Saved: {path}')

    # Final CR
    df = get_combinations_table_cr_with_ci('Final_Combinations', ds)
    if df is not None:
        path = os.path.join(output_dir, f'final_cr_{ds}_95ci.csv')
        df.to_csv(path, index=False)
        print(f'Saved: {path}')

Saved: figs/tables_CI/hpo_sa_brca_95ci.csv
Saved: figs/tables_CI/hpo_cr_brca_95ci.csv
Saved: figs/tables_CI/intermediate_sa_brca_95ci.csv
Saved: figs/tables_CI/intermediate_cr_brca_95ci.csv
Saved: figs/tables_CI/final_sa_brca_95ci.csv
Saved: figs/tables_CI/final_cr_brca_95ci.csv
Saved: figs/tables_CI/hpo_sa_lgg_95ci.csv
Saved: figs/tables_CI/hpo_cr_lgg_95ci.csv
Saved: figs/tables_CI/intermediate_sa_lgg_95ci.csv
Saved: figs/tables_CI/intermediate_cr_lgg_95ci.csv
Saved: figs/tables_CI/final_sa_lgg_95ci.csv
Saved: figs/tables_CI/final_cr_lgg_95ci.csv
